# Surface and Interior Meridional Circulation in the Sun — Implementation / 구현

**Paper**: Hanasoge, S. M., *Living Reviews in Solar Physics*, 19:3, 2022. DOI: 10.1007/s41116-022-00034-7

This notebook implements pedagogical models of solar meridional circulation (MC):
1. Axisymmetric meridional-flow profile (single-cell vs. two-cell in radius)
2. Mass-conservation constraint linking surface flow to return-flow speed
3. Travel-time perturbation schematic (time-distance helioseismology)
4. Ring-diagram geometry for patch-averaged flow estimation
5. Babcock-Leighton magnetic-surge advection from mid-latitude to pole
6. Solar-cycle time-dependence of MC amplitude

이 노트북은 태양 자오면 순환에 대한 교육용 모델을 구현한다: (1) 축대칭 자오면 흐름 프로파일, (2) 질량 보존 제약, (3) 왕복시간 섭동 모식도, (4) 링 다이어그램 기하, (5) Babcock-Leighton surge 이류, (6) 태양주기 시간 의존성.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Wedge
from matplotlib.colors import TwoSlopeNorm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

R_SUN = 6.957e8  # m, solar radius
V_SURF = 20.0    # m/s, typical surface poleward flow
OMEGA0 = 2.7e-6  # rad/s, rough sidereal rotation

## Part 1: Meridional Circulation Profile — Single vs. Two-Cell / 자오면 순환 프로파일 — 단일 셀 vs. 이중 셀

We model the stream function $\psi(r,\theta)$ and derive $v_r, v_\theta$ from $\mathbf{v} = \nabla\times(\psi\hat{\mathbf{e}}_\phi)$. This ensures exact mass conservation (anelastic) for a chosen density profile $\rho(r)$.

스트림 함수 $\psi(r,\theta)$로부터 $\mathbf{v}=\nabla\times(\psi\hat{\mathbf{e}}_\phi)$를 통해 자동 질량보존을 만족하는 MC를 정의한다. 단일 셀(single-cell)과 반경 방향 두 셀(two-cell in radius) 두 가지 시나리오를 비교한다.

In [ ]:
def density_profile(r_over_Rsun):
    """Rough power-law density stratification in the convection zone.

    Args:
        r_over_Rsun: Normalized radius r/R_sun, array-like in [0.7, 1.0].

    Returns:
        Density normalized so that rho=1 at r=R_sun; increases ~10^3 to base of CZ.
    """
    # Empirical: rho(0.7)/rho(1.0) ~ 1000 in solar convection zone.
    # Model rho ~ (1/r - c)^3 tuned to give ~1000x contrast.
    alpha = 8.0
    return (1.0 / r_over_Rsun) ** alpha * np.exp(alpha * (1 - 1/r_over_Rsun) * 0)


def stream_single_cell(r, theta, r_bottom=0.70, r_top=1.0):
    """Single-cell stream function psi(r, theta).

    Produces poleward flow at top, equatorward return at bottom, closed loop.
    """
    R = (r - r_bottom) / (r_top - r_bottom)
    # Zero at r=r_bottom and r=r_top, peaks mid-depth.
    radial = np.sin(np.pi * R) * (R > 0) * (R < 1)
    lat = np.sin(theta) ** 2 * np.cos(theta)  # ~sin^2 cos: zero at pole and equator
    return radial * lat


def stream_two_cell(r, theta, r_bottom=0.70, r_top=1.0, r_mid=0.90):
    """Two-cell (in radius) stream function."""
    R_lower = (r - r_bottom) / (r_mid - r_bottom)
    R_upper = (r - r_mid) / (r_top - r_mid)
    lower = np.sin(np.pi * R_lower) * (R_lower > 0) * (R_lower < 1)
    upper = -np.sin(np.pi * R_upper) * (R_upper > 0) * (R_upper < 1)
    lat = np.sin(theta) ** 2 * np.cos(theta)
    return (lower + upper) * lat


def flow_from_stream(psi_func, r_grid, theta_grid):
    """Compute (v_r, v_theta) from stream function via finite differences.

    v_r = (1 / (rho r sin theta)) d(psi sin theta)/dtheta
    v_theta = -(1 / (rho r)) d(r psi)/dr
    Here we use rho=1 for simplicity in showing shape.
    """
    R, TH = np.meshgrid(r_grid, theta_grid, indexing='ij')
    psi = psi_func(R, TH)
    rho = density_profile(R)
    # Partial derivatives via central differences.
    dpsi_dtheta = np.gradient(psi * np.sin(TH), theta_grid, axis=1)
    dpsi_dr = np.gradient(R * psi, r_grid, axis=0)
    v_r = dpsi_dtheta / (rho * R * np.sin(TH) + 1e-12)
    v_th = -dpsi_dr / (rho * R + 1e-12)
    return v_r, v_th, psi


r = np.linspace(0.70, 1.0, 80)
theta = np.linspace(np.deg2rad(1), np.deg2rad(179), 120)
vr_s, vt_s, psi_s = flow_from_stream(stream_single_cell, r, theta)
vr_d, vt_d, psi_d = flow_from_stream(stream_two_cell, r, theta)

# Normalize v_theta at the surface to match ~20 m/s at 45 deg latitude.
i_surf = -1
j_mid = len(theta) // 2 - len(theta) // 4  # ~45 deg colatitude
scale_s = V_SURF / max(abs(vt_s[i_surf, j_mid]), 1e-6)
scale_d = V_SURF / max(abs(vt_d[i_surf, j_mid]), 1e-6)
vr_s, vt_s = vr_s * scale_s, vt_s * scale_s
vr_d, vt_d = vr_d * scale_d, vt_d * scale_d

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(projection='polar'))
for ax, vth, title in zip(axes, [vt_s, vt_d], ['Single-cell / 단일 셀', 'Two-cell (radius) / 반경 방향 이중 셀']):
    TH, R = np.meshgrid(theta, r, indexing='xy')
    c = ax.pcolormesh(TH.T, R.T, vth, cmap='RdBu_r', shading='auto',
                     norm=TwoSlopeNorm(vmin=-25, vcenter=0, vmax=25))
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_rlim(0.70, 1.0)
    ax.set_title(title + '\n(colour: v_theta; red=equatorward, blue=poleward)')
    plt.colorbar(c, ax=ax, shrink=0.7, label='v_theta [m/s]')
plt.tight_layout()
plt.show()

## Part 2: Mass Conservation — Return-Flow Speed / 질량 보존 — 귀환 흐름 속도

From $\nabla\cdot(\rho\mathbf{v}_m)=0$, the surface flow $v_s$ feeding a deeper return flow $v_r$ satisfies approximately
$$\rho_s v_s A_s \approx \rho_r v_r A_r$$
so the return-flow speed is
$$v_r \approx v_s \frac{\rho_s A_s}{\rho_r A_r}.$$
For a thin shell, $A_r/A_s = (r_r/r_s)^2$.

질량 보존에서 얕은 층(밀도 낮음, 단면적 큼)의 빠른 흐름이 깊은 층(밀도 크고 단면적 작음)의 훨씬 느린 흐름으로 변환된다. 이 변환 비율은 $\rho$ 구조에 민감하여 return flow 깊이 $r_r$에 강하게 의존한다.

In [ ]:
def return_flow_speed(v_surface, r_surface, r_return, rho_surface, rho_return):
    """Mass-conservation estimate of return-flow speed.

    Args:
        v_surface: Surface poleward speed in m/s.
        r_surface: Radius of surface layer in solar radii.
        r_return: Radius of return layer in solar radii.
        rho_surface: Density at surface (normalized).
        rho_return: Density at return depth (normalized).

    Returns:
        Estimated return-flow speed in m/s.
    """
    area_ratio = (r_surface / r_return) ** 2
    return v_surface * (rho_surface / rho_return) * area_ratio


# Standard solar model density ratios (approximate)
depths = np.linspace(0.70, 0.99, 50)
rho_profile = density_profile(depths)
rho_surface = density_profile(1.0)

v_return = np.array([return_flow_speed(V_SURF, 1.0, d, rho_surface, density_profile(d))
                      for d in depths])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.semilogy(depths, density_profile(depths) / rho_surface, 'b-', lw=2)
ax1.set_xlabel('r / R_sun')
ax1.set_ylabel(r'$\rho(r)/\rho_{surface}$')
ax1.set_title('Density stratification / 밀도 층화')
ax1.grid(True, which='both', alpha=0.3)

ax2.plot(depths, v_return, 'r-', lw=2, label='Mass-conservation estimate')
ax2.axhspan(2, 5, color='green', alpha=0.2, label='Observed return flow range (2–5 m/s)')
ax2.axvline(0.90, color='k', ls='--', alpha=0.5, label='Candidate return depth 0.9')
ax2.axvline(0.77, color='purple', ls='--', alpha=0.5, label='Mean-field prediction 0.77')
ax2.set_xlabel('Return depth r / R_sun')
ax2.set_ylabel('Return-flow speed [m/s]')
ax2.set_title('Return-flow speed vs. depth / 귀환 흐름 속도 vs. 깊이')
ax2.set_yscale('log')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print('Numerical estimates of return-flow speed:')
for d in [0.95, 0.90, 0.85, 0.80, 0.77, 0.70]:
    v = return_flow_speed(V_SURF, 1.0, d, rho_surface, density_profile(d))
    print(f'  Return depth {d:.2f} R_sun:  v_return ~ {v:.2f} m/s')

## Part 3: Travel-Time Perturbation Schematic / 왕복시간 섭동 모식도

Time-distance helioseismology measures the cross-correlation of surface wave fields at two points. A flow $\mathbf{v}$ along the ray path induces a travel-time perturbation
$$\delta\tau = -2\int_\Gamma \hat{\mathbf{n}}\cdot\mathbf{v}/c^2\,ds.$$

We illustrate this with a simple parabolic ray path through a flow-bearing slab.

두 표면점 사이의 파동 cross-correlation에서 흐름에 의한 왕복시간 차이를 계산한다. 순방향/역방향 비대칭이 $\delta\tau$로 나타나 MC 크기를 추정한다.

In [ ]:
def ray_path(x_start, x_end, depth, n_pts=200):
    """Generate a parabolic ray path between two surface points.

    Args:
        x_start, x_end: Horizontal (meridional) endpoints.
        depth: Maximum depth reached by the ray.
        n_pts: Number of sample points.

    Returns:
        (x, z) arrays describing the ray path.
    """
    t = np.linspace(0, 1, n_pts)
    x = x_start + (x_end - x_start) * t
    z = -4 * depth * t * (1 - t)  # parabolic dip, negative = below surface
    return x, z


def travel_time_perturbation(x_path, z_path, flow_func, c_sound=7000.0):
    """Compute delta_tau via line integral -2 * n_hat . v / c^2 ds.

    Args:
        x_path, z_path: Ray coordinates in meters.
        flow_func: Function returning (v_x, v_z) at (x, z) in m/s.
        c_sound: Representative sound speed in m/s.

    Returns:
        Travel-time perturbation in seconds.
    """
    dx = np.diff(x_path)
    dz = np.diff(z_path)
    ds = np.sqrt(dx ** 2 + dz ** 2)
    nx = dx / (ds + 1e-12)
    nz = dz / (ds + 1e-12)
    x_mid = 0.5 * (x_path[:-1] + x_path[1:])
    z_mid = 0.5 * (z_path[:-1] + z_path[1:])
    vx, vz = flow_func(x_mid, z_mid)
    integrand = (nx * vx + nz * vz) / c_sound ** 2
    return -2.0 * np.sum(integrand * ds)


def mc_like_flow(x, z):
    """Toy MC: poleward (positive x) near surface, equatorward in deep layers."""
    # Map z in [-depth, 0] to [0, 1]; positive flow near surface, negative deep.
    depth_scale = 2.0e8  # m
    profile = np.tanh(z / depth_scale + 0.5)  # ~+1 at surface, ~-1 deep
    vx = V_SURF * profile
    vz = np.zeros_like(x)
    return vx, vz


# Compare northward vs southward going rays of distance 6 deg arc.
dist = np.deg2rad(6.0) * R_SUN  # 6 deg travel distance
depth = 1.0e8  # 100 Mm depth of ray bottom

xN, zN = ray_path(0.0, dist, depth)     # northward-going ray
xS, zS = ray_path(dist, 0.0, depth)     # southward-going ray

dtau_N = travel_time_perturbation(xN, zN, mc_like_flow)
dtau_S = travel_time_perturbation(xS, zS, mc_like_flow)
dtau_NS = dtau_N - dtau_S

print(f'Northward-going travel-time perturbation:  {dtau_N*1000:.3f} ms')
print(f'Southward-going travel-time perturbation:  {dtau_S*1000:.3f} ms')
print(f'N-S difference (sensitive to MC):           {dtau_NS*1000:.3f} ms')
print(f'(Typical observed N-S delta_tau is ~0.5–2 s, depending on travel distance and latitude.)')

fig, ax = plt.subplots(figsize=(10, 5))
# Plot flow background.
xx = np.linspace(0, dist, 60)
zz = np.linspace(-1.5 * depth, 0, 40)
XX, ZZ = np.meshgrid(xx, zz)
VX, _ = mc_like_flow(XX, ZZ)
im = ax.pcolormesh(XX / 1e6, ZZ / 1e6, VX, cmap='RdBu_r', shading='auto',
                   norm=TwoSlopeNorm(vmin=-V_SURF, vcenter=0, vmax=V_SURF))
plt.colorbar(im, ax=ax, label='v_x [m/s]  (poleward → positive)')
ax.plot(xN / 1e6, zN / 1e6, 'k-', lw=2, label='Ray path')
ax.plot([xN[0] / 1e6, xN[-1] / 1e6], [0, 0], 'ko', markersize=8)
ax.set_xlabel('Distance along meridian [Mm]')
ax.set_ylabel('Depth [Mm]')
ax.set_title('Ray path through MC field — travel-time is asymmetric N vs S\n'
             '광선 경로와 MC — 남북 방향에서 travel time이 비대칭')
ax.legend()
ax.axhline(0, color='k', lw=0.5)
plt.tight_layout()
plt.show()

## Part 4: Ring-Diagram Geometry / 링 다이어그램 기하

RDA (Hill 1988) tiles the solar disk into $15°\times 15°$ patches and fits high-degree $p$-mode power spectra to a dispersion relation shifted by the patch-averaged flow:
$$\omega_{obs}(\mathbf{k}) = \omega_0(k) + \mathbf{k}\cdot\mathbf{U}_{patch}.$$
A ring in the $(k_x, k_y)$ plane at fixed $\omega$ is displaced off-centre by $\mathbf{U}$.

RDA는 태양 원반을 $15°\times 15°$ 패치로 나눠 고차 $p$-모드 dispersion의 중심 이동을 측정해 patch-평균 수평 흐름을 얻는다.

In [ ]:
def ring_power_spectrum(k_x, k_y, U_patch, omega_fixed=np.deg2rad(3.0), k0=0.3, width=0.03):
    """Power at fixed omega over (kx, ky) plane; ring displaced by U.

    Args:
        k_x, k_y: Wavenumber components (normalized).
        U_patch: (Ux, Uy) horizontal flow, units chosen to shift in k-space.
        omega_fixed: Fixed angular frequency slice (arb).
        k0: Nominal ring radius.
        width: Ring width.

    Returns:
        2D power spectrum array.
    """
    Ux, Uy = U_patch
    k_eff = np.sqrt((k_x - Ux) ** 2 + (k_y - Uy) ** 2)
    return np.exp(-((k_eff - k0) / width) ** 2)


kx = np.linspace(-0.5, 0.5, 200)
ky = np.linspace(-0.5, 0.5, 200)
KX, KY = np.meshgrid(kx, ky)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, U, title in zip(axes,
                        [(0.0, 0.0), (0.05, 0.0), (0.0, 0.08)],
                        ['No flow / 흐름 없음', 'Differential rotation / 차등자전 (kx shift)', 'MC / 자오면 순환 (ky shift)']):
    P = ring_power_spectrum(KX, KY, U)
    im = ax.pcolormesh(KX, KY, P, cmap='magma', shading='auto')
    # Overlay nominal ring center
    ax.plot(U[0], U[1], 'c+', markersize=20, mew=2, label=f'Centre = U={U}')
    ax.plot(0, 0, 'w.', markersize=8, label='Origin')
    ax.set_xlabel('k_x (zonal)')
    ax.set_ylabel('k_y (meridional)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.legend(loc='lower left', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print('Ring-diagram principle: the centre of the ring in (kx, ky) at fixed omega')
print('is displaced from the origin by an amount proportional to the patch-averaged flow.')

## Part 5: Babcock-Leighton Surge — Mid-Latitude to Pole / 중위도에서 극까지의 자기 surge

In the flux-transport dynamo (Babcock 1961; Wang, Nash & Sheeley 1989; Choudhuri et al. 1995), leading/following polarity asymmetry of active regions emerging at mid-latitude is carried poleward by surface MC, cancelling the old polar field and building the new-cycle field. We simulate a pulse of magnetic flux released at latitude $30°$ being advected by a prescribed poleward flow $v(\theta)$.

Flux-transport dynamo: 중위도 활동영역에서 방출된 leading/following 극성 비대칭이 표면 MC에 의해 극으로 수송. 이 '선속 surge'를 1차원 이류 방정식 $\partial_t B + v(\theta)\partial_\theta B = \eta\partial_\theta^2 B$로 모형화.

In [ ]:
def advect_diffuse_1d(B0, theta, v, eta, t_end, dt):
    """1D advection-diffusion of magnetic flux along colatitude.

    Args:
        B0: Initial field profile.
        theta: Colatitude grid in radians.
        v: Meridional velocity profile v(theta) in rad/s (sign: negative=toward pole).
        eta: Turbulent diffusion coefficient in rad^2/s.
        t_end: End time in seconds.
        dt: Time step in seconds.

    Returns:
        List of (time, profile) snapshots.
    """
    B = B0.copy()
    dtheta = theta[1] - theta[0]
    n_steps = int(t_end / dt)
    snap_every = max(n_steps // 5, 1)
    snapshots = [(0.0, B.copy())]
    for step in range(1, n_steps + 1):
        dB_dtheta = np.gradient(B, dtheta)
        d2B_dtheta2 = np.gradient(dB_dtheta, dtheta)
        B = B - dt * v * dB_dtheta + dt * eta * d2B_dtheta2
        # Zero-flux BCs at poles/equator
        B[0] = B[1]
        B[-1] = B[-2]
        if step % snap_every == 0:
            snapshots.append((step * dt, B.copy()))
    return snapshots


# Colatitude from equator (pi/2) to pole (0)
theta = np.linspace(np.deg2rad(1), np.deg2rad(89), 200)  # northern hemisphere only
lat_deg = 90 - np.rad2deg(theta)

# Poleward flow: v(theta) in rad/s negative toward pole (theta decreasing)
v_peak = V_SURF / R_SUN  # angular speed, rad/s
v_profile = -v_peak * np.sin(2 * theta)  # peaks at theta=pi/4 (45 deg latitude)

# Initial pulse at 30 deg latitude
theta0 = np.deg2rad(60)  # colatitude 60 deg = latitude 30 deg
B0 = np.exp(-((theta - theta0) / np.deg2rad(5)) ** 2)

eta = 5e3 ** 2 / R_SUN ** 2  # rough turbulent diffusion, rad^2/s
# Total evolution time: ~1 year
t_year = 365.25 * 86400.0
dt = 3600.0 * 24  # 1 day

snaps = advect_diffuse_1d(B0, theta, v_profile, eta, 1.5 * t_year, dt)

fig, ax = plt.subplots(figsize=(11, 6))
for i, (t, B) in enumerate(snaps):
    ax.plot(lat_deg, B, lw=2, label=f't = {t / t_year:.2f} yr')
ax.set_xlabel('Latitude [deg]')
ax.set_ylabel('Magnetic flux (arb. units)')
ax.set_title('Babcock-Leighton surge: flux advected from 30° to pole by surface MC\n'
             'Babcock-Leighton surge: 표면 MC에 의한 30° → 극 자기 선속 수송')
ax.axvline(30, color='k', ls=':', alpha=0.5, label='Emergence latitude')
ax.axvline(75, color='r', ls=':', alpha=0.5, label='High-latitude destination')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Time for a flux element to traverse 30° → 75°
delta_theta = np.deg2rad(75 - 30)
avg_v = np.mean(np.abs(v_profile[(lat_deg > 30) & (lat_deg < 75)]))
t_cross = delta_theta / avg_v
print(f'Characteristic advection time 30 → 75 deg:  {t_cross / t_year:.2f} yr')

## Part 6: Solar-Cycle Time-Dependence of MC / 태양주기에 따른 MC 변화

Observations (Hathaway & Rightmire 2010; Komm et al. 2018) reveal that the surface MC amplitude anti-correlates with sunspot number: MC is larger at minima and smaller at maxima, with ~2–4 m/s variation. We parameterize:
$$V_{MC}(t) = V_0 - \Delta V\cdot S(t)/S_{max},$$
where $S(t)$ is the sunspot number following an 11-yr cycle.

태양주기와 MC의 반상관 관계. MC는 minima에서 크고 maxima에서 작다 (~2-4 m/s 변동).

In [ ]:
def sunspot_number(t, cycle_period=11.0, S_max=150.0):
    """Schematic sunspot cycle: skewed sinusoid.

    Args:
        t: Time in years.
        cycle_period: Cycle length in years.
        S_max: Peak sunspot number.
    """
    phase = 2 * np.pi * t / cycle_period
    S = S_max * np.maximum(0.0, np.sin(phase)) ** 2
    return S


def mc_amplitude(t, V_0=16.0, delta_V=3.0, cycle_period=11.0, S_max=150.0):
    """Time-dependent surface MC amplitude (anti-correlated with sunspots)."""
    S = sunspot_number(t, cycle_period, S_max)
    return V_0 - delta_V * S / S_max


t_years = np.linspace(0, 22, 800)
S = sunspot_number(t_years)
V = mc_amplitude(t_years)

fig, ax1 = plt.subplots(figsize=(11, 5.5))
color1 = 'tab:red'
ax1.plot(t_years, S, color=color1, lw=2, label='Sunspot number')
ax1.set_xlabel('Time [years]')
ax1.set_ylabel('Sunspot number', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.fill_between(t_years, 0, S, alpha=0.2, color=color1)

ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.plot(t_years, V, color=color2, lw=2, label='Surface MC amplitude')
ax2.set_ylabel('MC amplitude [m/s]', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(10, 20)

plt.title('Solar-cycle anti-correlation: MC ↑ when sunspots ↓\n'
          '태양주기 반상관: MC는 sunspot이 낮을 때 크다')
fig.tight_layout()
plt.show()

# Report correlation coefficient
corr = np.corrcoef(S, V)[0, 1]
print(f'Modelled correlation coefficient (S vs V_MC):  {corr:.3f}  (expected: strong negative)')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Axisymmetric MC profile | Single vs. two-cell stream function | Gizon et al. (2020), Rajaguru & Antia (2020) inversions |
| Mass conservation / 질량 보존 | $\nabla\cdot(\rho\mathbf{v})=0$, anelastic | Stream-function inversions with mass conservation enforced |
| Travel-time perturbation | $\delta\tau \sim \int v/c^2 ds$ | Born-kernel TD inversions (Birch & Kosovichev 2001) |
| Ring-diagram geometry | $\omega_{obs}=\omega_0 + \mathbf{k}\cdot\mathbf{U}$ | HMI/GONG ring-diagram pipelines (Komm et al. 2015, 2018) |
| Babcock-Leighton surge | Mid-latitude → pole advection | Flux-transport dynamo models (Dikpati, Charbonneau 2020) |
| Cycle-dependent MC | Anti-correlation with sunspots | Hathaway & Rightmire (2010); Komm et al. (2018) |

### Key physical insights / 핵심 물리적 통찰

1. **Mass conservation** / 질량 보존 forces the return flow to be much slower than the surface flow (factor of density ratio ~10^3 if return is at base of CZ).
2. **Two-cell structures** / 두 셀 구조 inferred in TD analyses can be artifacts of mode-frequency-dependent centre-to-limb residuals (Rajaguru & Antia 2020).
3. **Advection timescale** / 이류 시간 규모 for Babcock-Leighton surge (30° → pole) is ~1 year at surface speeds — the 11-yr cycle period emerges from the full loop including deep return.
4. **Cycle modulation** / 주기 조절 of MC amplitude by 2–4 m/s provides dynamo feedback.